In [ ]:
!pip install -U albumentations==1.4.10 albucore==0.0.12 opencv-python==4.10.0.84 --quiet

In [ ]:
import os, random, warnings
from pathlib import Path
import numpy as np
from PIL import Image
import cv2
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from albumentations import (
    Compose, HorizontalFlip, RandomBrightnessContrast, GaussianBlur,
    ShiftScaleRotate, Resize, Normalize
)
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
VOC_ROOT = Path("/kaggle/input/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val")
IMG_ROOT = VOC_ROOT / "JPEGImages"
GT_ROOT  = VOC_ROOT / "SegmentationClass"
SPLIT_ROOT = VOC_ROOT / "ImageSets" / "Segmentation"

assert IMG_ROOT.exists(), IMG_ROOT
assert GT_ROOT.exists(), GT_ROOT
assert (SPLIT_ROOT / "train.txt").exists(), SPLIT_ROOT

def read_ids(p): 
    with open(p) as f: return [x.strip() for x in f if x.strip()]

train_ids = read_ids(SPLIT_ROOT / "train.txt")        # 1464
val_ids   = read_ids(SPLIT_ROOT / "val.txt")          # 1449
print("IDs → train:", len(train_ids), "val:", len(val_ids))

BASE_OUT   = Path("/kaggle/working/outputs")
SEEDS_DIR  = BASE_OUT / "seed_overlays"          # optional (visuals)
PSEUDO_DIR = BASE_OUT / "pseudo_masks" / "gradcam"
EXP_DIR    = BASE_OUT / "B_dice"
for d in [SEEDS_DIR, PSEUDO_DIR, EXP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Saving pseudo masks to:", PSEUDO_DIR)

In [ ]:
import torchvision
from torchvision.models import resnet50, ResNet50_Weights

class CAMHelper:
    def __init__(self):
        try:
            m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        except Exception as e:
            print("WARNING: pretrained weights unavailable; using random weights (CAM may be useless).", e)
            m = resnet50(weights=None)
        m.eval().to(DEVICE)
        self.model = m
        self.feats, self.grads = [], []
        def f_hook(_, __, out): self.feats.append(out.detach())
        def b_hook(_, grad_in, grad_out): self.grads.append(grad_out[0].detach())
        self.handles = [
            m.layer4[-1].conv3.register_forward_hook(f_hook),
            m.layer4[-1].conv3.register_full_backward_hook(b_hook),
        ]
        self.pre = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.485,0.456,0.406],
                                             std =[0.229,0.224,0.225])
        ])
    @torch.no_grad()
    def encode(self, pil_img):
        x = self.pre(pil_img).unsqueeze(0).to(DEVICE)
        out = self.model(x)                 # forward to fill feats
        return out

    def __call__(self, pil_img):
        self.feats.clear(); self.grads.clear()
        x = self.pre(pil_img).unsqueeze(0).to(DEVICE)
        x.requires_grad_(True)
        logits = self.model(x)
        cls = logits.argmax(dim=1)
        logits[0, cls].backward()
        A = self.feats[-1][0]          # [C,H,W]
        G = self.grads[-1][0]          # [C,H,W]
        w = G.mean(dim=(1,2))          # [C]
        cam = torch.relu((w[:,None,None] * A).sum(0))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        return cam.detach().cpu().numpy()

    def close(self):
        for h in self.handles: h.remove()

cam_helper = CAMHelper()


def seeds_to_pseudo(ids, th=0.30):
    wrote = 0
    for k, img_id in enumerate(ids, 1):
        ip = IMG_ROOT / f"{img_id}.jpg"
        if not ip.exists(): ip = IMG_ROOT / f"{img_id}.jpeg"
        img = Image.open(ip).convert("RGB")
        W,H = img.size
        cam = cam_helper(img)                            # [h,w] 0..1
        cam_up = cv2.resize(cam, (W,H), interpolation=cv2.INTER_LINEAR)
        mask = (cam_up >= th).astype(np.uint8)*255
        (PSEUDO_DIR / f"{img_id}.png").parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(mask).save(PSEUDO_DIR / f"{img_id}.png")
        # optional quick overlay samples
        if k <= 8:
            arr = np.array(img).copy()
            overlay = arr.copy()
            overlay[mask>0] = (0.4*overlay[mask>0] + 0.6*np.array([0,255,0])).astype(np.uint8)
            Image.fromarray(overlay).save(SEEDS_DIR / f"{img_id}_overlay.png")
        wrote += 1
        if k % 200 == 0: print(f"  built {k}/{len(ids)}...")
    print(f"Done. Pseudo masks written: {wrote} → {PSEUDO_DIR}")


seeds_to_pseudo(train_ids, th=0.30)

paths = sorted(PSEUDO_DIR.glob("*.png"))
nonempty = 0
for p in paths[:100]:
    if np.array(Image.open(p)).max() > 0: nonempty += 1
print("num masks:", len(paths), "| non-empty in first 100:", nonempty)
assert len(paths) >= 1000 and nonempty > 0, "Pseudo generation failed; CAM likely uninitialized."

In [ ]:
IGNORE_IDX = 255
IMG_SIZE   = 256
BATCH_TRAIN = 8
BATCH_VAL   = 8

class VOCPseudoBinary(Dataset):
    def __init__(self, ids, img_root, pseudo_root, gt_root, train=True, size=256):
        self.ids = list(ids)
        self.img_root, self.pseudo_root, self.gt_root = map(Path, [img_root, pseudo_root, gt_root])
        self.train = bool(train)
        self.size  = int(size)

        aug_train = Compose([
            HorizontalFlip(p=0.5),
            RandomBrightnessContrast(p=0.2),
            GaussianBlur(blur_limit=(3,5), p=0.15),
            ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15,
                             border_mode=cv2.BORDER_CONSTANT, p=0.5),
            Resize(self.size, self.size),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        aug_val = Compose([
            Resize(self.size, self.size),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        self.tr = aug_train if self.train else aug_val

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]
        ip = self.img_root / f"{img_id}.jpg"
        if not ip.exists(): ip = self.img_root / f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB"))
        H,W = img.shape[:2]

        pp = self.pseudo_root / f"{img_id}.png"
        if pp.exists():
            pm = np.array(Image.open(pp))
            if pm.ndim == 3: pm = pm[...,0]
        else:
            pm = np.zeros((H,W), np.uint8)

        gp = self.gt_root / f"{img_id}.png"
        if gp.exists():
            gt = np.array(Image.open(gp))
        else:
            gt = np.full((H,W), IGNORE_IDX, np.uint8)

        if pm.shape != (H,W): pm = cv2.resize(pm, (W,H), interpolation=cv2.INTER_NEAREST)
        if gt.shape != (H,W): gt = cv2.resize(gt, (W,H), interpolation=cv2.INTER_NEAREST)

        gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))

        out = self.tr(image=img, mask=pm, masks=[gbin])
        x   = out["image"].float()
        pm2 = torch.as_tensor(out["mask"]).squeeze()
        gt2 = torch.as_tensor(out["masks"][0]).squeeze()

        # robust binarization for pseudo
        g = (pm2 > (0.5 if pm2.dtype.is_floating_point else 0)).long()
        q = gt2.long()   # eval

        return x, g, q, img_id

def make_loaders(num_workers=2, pin=True):
    train_ds = VOCPseudoBinary(train_ids, IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=True,  size=IMG_SIZE)
    val_ds   = VOCPseudoBinary(val_ids,   IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=False, size=IMG_SIZE)
    train_dl = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True,  num_workers=num_workers, pin_memory=pin)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_VAL,   shuffle=False, num_workers=num_workers, pin_memory=pin)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()

fg_pix = 0; tot_pix = 0
for step,(x,g,_,_) in enumerate(train_dl):
    fg_pix += (g>0).sum().item()
    tot_pix += g.numel()
    if step==9: break
print(f"[probe] pseudo>0 pixels: {fg_pix} ({fg_pix/max(1,tot_pix):.2%})")
assert fg_pix>0, "Pseudo labels are all background — check PSEUDO_DIR."

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50

def build_deeplab_binary(num_classes=2):
    m = deeplabv3_resnet50(weights=None, weights_backbone=None)
    in_ch = m.classifier[4].in_channels
    m.classifier[4] = nn.Conv2d(in_ch, num_classes, kernel_size=1)
    if m.aux_classifier is not None:
        in_aux = m.aux_classifier[4].in_channels
        m.aux_classifier[4] = nn.Conv2d(in_aux, num_classes, kernel_size=1)
    return m

model = build_deeplab_binary(2).to(DEVICE)

In [ ]:
from torch.cuda.amp import GradScaler, autocast
import numpy as np

@torch.no_grad()
def estimate_fg_ratio(dloader, max_batches=20):
    fg = tot = 0
    for b,(_,g,_,_) in enumerate(dloader):
        fg  += (g>0).sum().item()
        tot += g.numel()
        if b+1>=max_batches: break
    return max(fg/tot, 1e-6)

def dice_loss_logits(logits, target_01, eps=1e-6):
    probs = torch.softmax(logits, dim=1)[:,1:2]
    target = target_01.float()
    inter = (probs*target).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (union + eps)
    return 1 - dice.mean()

def fast_confusion(pred, target, num_classes=2, ignore_index=255):
    mask = target != ignore_index
    pred = pred[mask]; target = target[mask]
    m = pred * num_classes + target
    cm = torch.bincount(m, minlength=num_classes*num_classes).reshape(num_classes, num_classes)
    return cm.cpu().numpy()

def iou_from_cm(cm):
    ious=[]
    for c in range(cm.shape[0]):
        tp   = cm[c,c]
        denom= cm[c,:].sum()+cm[:,c].sum()-tp
        ious.append( (tp/denom) if denom>0 else 0.0 )
    return ious[0], ious[1], float(np.mean(ious))

fg_ratio = estimate_fg_ratio(train_dl)
w_bg = 1.0
w_fg = float(max(1.0, (1.0/fg_ratio) - 1.0))
ce_weight = torch.tensor([w_bg, w_fg], dtype=torch.float32, device=DEVICE)
print(f"CE class weights → bg={w_bg:.2f}, fg={w_fg:.2f}")

ce = nn.CrossEntropyLoss(weight=ce_weight)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = GradScaler()

def train_one_epoch():
    model.train()
    total = 0.0

    for step, (x, g, _, _) in enumerate(train_dl, 1):
        x = x.to(DEVICE, non_blocking=True)
        y = g.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with autocast():
            out  = model(x)["out"]
            loss = ce(out, y) + 0.5 * dice_loss_logits(out, y.unsqueeze(1))

        # Correct AMP sequence
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        total += loss.item() * x.size(0)

        if step % 50 == 0 or step == len(train_dl):
            print(f"  step {step:4d}/{len(train_dl)} | loss {loss.item():.4f}")

    return total / len(train_dl.dataset)

@torch.no_grad()
def evaluate(dloader):
    model.eval()
    cm = np.zeros((2,2),dtype=np.float64)
    for x,_,q,_ in dloader:
        x=x.to(DEVICE,non_blocking=True)
        pred = model(x)["out"].argmax(1).cpu()
        cm += fast_confusion(pred, q)
    bg, fg, miou = iou_from_cm(cm)
    return {"IoU_bg":bg, "IoU_fg":fg, "mIoU":miou}

In [ ]:
EPOCHS=15; PATIENCE=3
CKPT_PATH = EXP_DIR / "deeplab_binary_dice_best.pth"

best=0.0; wait=0
for ep in range(1, EPOCHS+1):
    tl = train_one_epoch()
    mets = evaluate(val_dl)
    print(f"Epoch {ep:02d} | loss={tl:.4f} | mIoU={mets['mIoU']:.3f} (bg={mets['IoU_bg']:.3f}, fg={mets['IoU_fg']:.3f})")
    if mets["mIoU"] > best + 1e-4:
        best = mets["mIoU"]; wait=0
        torch.save(model.state_dict(), CKPT_PATH)
        print("  New best; checkpoint saved:", CKPT_PATH.name)
    else:
        wait += 1
        if wait > PATIENCE:
            print("Early stop "); break
print("Best mIoU:", round(best,3))

In [ ]:
import os, random, numpy as np, cv2, pandas as pd, torch
from tqdm.auto import tqdm
from pathlib import Path

torch.backends.cudnn.benchmark = True
_rng = np.random.default_rng(123)  

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def to_uint8(img_t: torch.Tensor) -> np.ndarray:
    x = img_t.detach().cpu().permute(1,2,0).numpy()
    x = (x * _STD + _MEAN) * 255.0
    return np.clip(x, 0, 255).astype(np.uint8)

def to_tensor_norm(img_u8: np.ndarray) -> torch.Tensor:
    x = img_u8.astype(np.float32) / 255.0
    x = (x - _MEAN) / _STD
    return torch.from_numpy(x).permute(2,0,1).contiguous()

def perturbations(img_u8: np.ndarray) -> dict:
    out = {}
    out['clean']      = img_u8
    out['blur']       = cv2.GaussianBlur(img_u8, (5,5), 1.0)
    out['brightness'] = np.clip(img_u8.astype(np.float32) * 1.25, 0, 255).astype(np.uint8)
    n = img_u8.astype(np.float32) + _rng.normal(0, 25, img_u8.shape).astype(np.float32)
    out['gauss']      = np.clip(n, 0, 255).astype(np.uint8)
    out['hflip']      = img_u8[:, ::-1]
    H, W = img_u8.shape[:2]
    M = cv2.getRotationMatrix2D((W/2, H/2), 15, 1.0)
    out['rotation']   = cv2.warpAffine(img_u8, M, (W, H), flags=cv2.INTER_LINEAR,
                                       borderMode=cv2.BORDER_REFLECT_101)
    return out

def fast_confusion(pred: torch.Tensor, target: torch.Tensor,
                   num_classes: int = 2, ignore_index: int = 255) -> torch.Tensor:
    pred = pred.view(-1).long().cpu()
    tgt  = target.view(-1).long().cpu()
    m = tgt != ignore_index
    pred = pred[m]; tgt = tgt[m]
    k = (tgt * num_classes + pred).to(torch.int64)
    hist = torch.bincount(k, minlength=num_classes*num_classes)
    return hist.view(num_classes, num_classes)

def iou_from_cm(cm: torch.Tensor) -> tuple[float, list[float]]:
    cm = cm.float()
    d = torch.diag(cm)
    den = cm.sum(1) + cm.sum(0) - d + 1e-7
    ious = (d / den).tolist()
    return float(np.mean(ious)), ious

@torch.inference_mode()
def evaluate_under_perturbations(
    val_loader,
    perturb_list=('clean','blur','brightness','gauss','hflip','rotation'),
    log_every=25,            
    max_batches=None,        
    sample_ratio=1.0):       
    model.eval()

    base_ds = val_loader.dataset
    if sample_ratio < 1.0:
        idxs = np.linspace(0, len(base_ds)-1, max(1, int(len(base_ds)*sample_ratio))).astype(int).tolist()
        sub = torch.utils.data.Subset(base_ds, idxs)
        vloader = torch.utils.data.DataLoader(
            sub, batch_size=val_loader.batch_size, shuffle=False,
            num_workers=val_loader.num_workers, pin_memory=getattr(val_loader, "pin_memory", False)
        )
        total_batches = len(vloader)
    else:
        vloader = val_loader
        total_batches = len(vloader)

    cms = {n: torch.zeros(2,2) for n in perturb_list}
    pbar = tqdm(vloader, total=total_batches, desc="robust-eval", leave=False)
    for b_idx, (x, g, q, _) in enumerate(pbar, 1):
        imgs_u8 = [to_uint8(xi) for xi in x]  # de-normalize once per batch
        # loop perturbations
        for n in perturb_list:
            batch = torch.stack([to_tensor_norm(perturbations(img)[n]) for img in imgs_u8], 0).to(DEVICE)
            pred = model(batch)['out'].argmax(1).cpu()
            for i in range(pred.size(0)):
                cms[n] += fast_confusion(pred[i], q[i], num_classes=2, ignore_index=IGNORE_IDX)

        if b_idx % log_every == 0 or b_idx == 1:
            msg = []
            for n in perturb_list:
                mi, ious = iou_from_cm(cms[n])
                msg.append(f"{n}:mIoU={mi:.3f}|bg={ious[0]:.3f}|fg={ious[1]:.3f}")
            pbar.set_postfix_str(" | ".join(msg))

        if (max_batches is not None) and (b_idx >= max_batches):
            break

    rows = []
    for n in perturb_list:
        mi, ious = iou_from_cm(cms[n])
        rows.append({'perturb': n, 'IoU_bg': round(ious[0],6), 'IoU_fg': round(ious[1],6), 'mIoU': round(mi,6)})

    df = pd.DataFrame(rows).set_index('perturb').reindex(list(perturb_list)).reset_index()
    out_csv = Path(EXP_DIR) / 'voc_val_robustness.csv'
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    display(df)
    return df

df_rob = evaluate_under_perturbations(val_dl)